# Run 3 — Fake-PD-Only (Zero-Shot) on the 17-Patient Test Set

Every Run 1 / Run 2 experiment trained on fake PD **plus 13 real PD patients**. In that mixed
regime the model has already seen the real distribution, so the fake→real gap is only ~0.02 and
each idea was competing for ~0.01 against a 0.0226 noise floor — undetectable by construction.

Run 3 removes real PD from training **and** from checkpoint selection. From
`docs/DEBUG_HANDOFF.md` §5.1 / §17.7:

| Regime | Real-PD Dice |
|---|---|
| fake-only (`run_002_v2`, best) | 0.695 |
| fake + 15 real (`v7_mixed`) | 0.781 |

That reopens a **0.09–0.15 domain gap** — 4–6× the old floor — so an idea that closes even a
third of it is now measurable.

**Two things differ from the Phase 5 notebooks and must not be carried over:**

1. **The 0.0226 noise floor does NOT apply here.** It was measured in the mixed regime. Section 2
   below measures the Run 3 floor directly from `base` seed42 vs seed7, and every verdict uses
   that measured value.
2. **`history.csv` has a different schema.** Selection is `dice_fake_va_smooth` (a 3-epoch
   trailing mean of fake-val Dice — what `ckpt_best.pth` was actually chosen on). `dice_real_va_diag`
   is a *diagnostic only*: real PD never touched a gradient or the selection rule.

In [ ]:
import os
import glob
import re
import csv

import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
import matplotlib.pyplot as plt
from scipy.ndimage import zoom

In [ ]:
BASE = "/Users/anshikabajpai/Desktop/github/RegGAN_domain_adaptation_v2"
RES  = f"{BASE}/results"
RUNS = f"{BASE}/segmentation_runs"

GT_DIR_D1  = "/Users/anshikabajpai/Desktop/AImed-lab/SEGMENTATIONS/PD-segmentations-final"
GT_DIR_NEW = "/Users/anshikabajpai/Downloads/final-segmentations"   # NOT "segmentations" -- landmine

# label -> (17pt prediction dir, run dir holding history.csv)
BASE_LABEL = "base seed42 (fake-only reference)"
SEED7_LABEL = "base seed7 (noise-floor pair)"

RUNS_INFO = {
    BASE_LABEL:                 (f"{RES}/real_pd_predictions_fakeonly_base_seed42_17pt", f"{RUNS}/run_fakeonly_base_seed42"),
    SEED7_LABEL:                (f"{RES}/real_pd_predictions_fakeonly_base_seed7_17pt",  f"{RUNS}/run_fakeonly_base_seed7"),
    "A3-none (control)":        (f"{RES}/real_pd_predictions_fakeonly_a3_none_17pt",     f"{RUNS}/run_fakeonly_a3_none_seed42"),
    "A3-up (end slices x2)":    (f"{RES}/real_pd_predictions_fakeonly_a3_up_17pt",       f"{RUNS}/run_fakeonly_a3_up_seed42"),
    "A3-down (end slices x0.3)":(f"{RES}/real_pd_predictions_fakeonly_a3_down_17pt",     f"{RUNS}/run_fakeonly_a3_down_seed42"),
    "E1 (position channel)":    (f"{RES}/real_pd_predictions_fakeonly_e1_17pt",          f"{RUNS}/run_fakeonly_e1_seed42"),
    "E2 (FiLM gating)":         (f"{RES}/real_pd_predictions_fakeonly_e2_17pt",          f"{RUNS}/run_fakeonly_e2_seed42"),
    "G2 (rim-darkness loss)":   (f"{RES}/real_pd_predictions_fakeonly_g2_17pt",          f"{RUNS}/run_fakeonly_g2_seed42"),
    "D (shape prior)":          (f"{RES}/real_pd_predictions_fakeonly_d_17pt",           f"{RUNS}/run_fakeonly_d_seed42"),
}
# Paths match the CKPT=/OUT_DIR= lines in slurm_files/fakeonly_*.sh and
# slurm_files/infer_fakeonly_*_17pt.sh -- not retyped from memory.

# Reference points from the MIXED regime, for context only. Never a Run 3 baseline.
MIXED_REPLICA_TEST   = 0.7780
MIXED_CROPSEQ_TEST   = 0.8057
MIXED_NOISE_FLOOR    = 0.0226

D1_PATIENTS = [
    "AC0D5A4D78B628", "AC0D7BF72F7712", "AC0D3459553205", "AC14D3737C0482",
    "AC19E7C19827FF", "AC149BC218E75C", "AC111633B463BB", "AC13300201B926",
    "AC12026D14291F", "AC13637DA25399",
]
NEW_PATIENTS = [
    "AC000550763509", "AC005135D3495B", "AC04433E37DB66", "AC056ADCE8BE28",
    "AC07607B9E5295", "AC0D1A9818A6FC", "AC0F11041F5180",
]
ALL_PATIENTS = D1_PATIENTS + NEW_PATIENTS
VAL_PATIENTS = ["AC0CE315D5758B", "AC0CEE9C24F2B7"]   # diagnostic only in Run 3

print(f"Test set: {len(ALL_PATIENTS)} patients (D1={len(D1_PATIENTS)}, new={len(NEW_PATIENTS)})")
print(f"Run 3: real PD never used for training or checkpoint selection.")

## Helpers — identical conventions to the Phase 5 notebooks

In [ ]:
def get_pid(fname):
    m = re.match(r"(AC[A-F0-9]+)", os.path.basename(fname), re.IGNORECASE)
    return m.group(1) if m else None

def patients_in_dir(directory):
    files = glob.glob(os.path.join(directory, "*.nii.gz"))
    return {get_pid(f): f for f in files if get_pid(f)}

def load_vol(path):
    return np.asarray(nib.load(path).get_fdata())

def load_nrrd_binary(path):
    """Verbatim from analysis_phase5_novel_approaches_17patients.ipynb."""
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)
    if arr.ndim == 4:
        arr = (arr.max(axis=0) > 0).astype(np.uint8)
        out = sitk.GetImageFromArray(arr)
        try:
            comp = sitk.VectorIndexSelectionCast(img, 0)
            out.CopyInformation(comp)
        except Exception:
            out.SetSpacing(img.GetSpacing()[:3]); out.SetOrigin(img.GetOrigin()[:3])
            out.SetDirection(img.GetDirection()[:9])
    else:
        arr = (arr > 0).astype(np.uint8)
        out = sitk.GetImageFromArray(arr)
        out.CopyInformation(img)
    try:
        out = sitk.DICOMOrient(out, "RAS")
    except Exception:
        pass
    sp  = out.GetSpacing()
    arr = sitk.GetArrayFromImage(out).astype(np.float32).transpose(2, 1, 0)
    fa  = float(sp[1]) / min(sp[1], sp[2])
    fs  = float(sp[2]) / min(sp[1], sp[2])
    if abs(fa - 1.0) > 0.02 or abs(fs - 1.0) > 0.02:
        arr = zoom(arr, (1.0, fa, fs), order=0)
    _, n_A, n_S = arr.shape
    if n_A != 384 or n_S != 384:
        arr = zoom(arr, (1.0, 384 / n_A, 384 / n_S), order=0)
    return (arr > 0.5).astype(bool)

def merge_meniscus(pred):
    return (pred > 0).astype(bool)

def dice_score(pred_bin, gt_bin):
    if pred_bin.shape != gt_bin.shape:
        raise ValueError(f"shape mismatch: pred {pred_bin.shape} vs gt {gt_bin.shape} "
                         f"-- refusing to silently resample, fix the source")
    inter = (pred_bin & gt_bin).sum()
    denom = pred_bin.sum() + gt_bin.sum()
    return float(2 * inter / denom) if denom > 0 else 1.0

## Section 1 — 17-patient test Dice

This is the zero-shot number: the checkpoint never saw a real-PD label during training **or**
selection, and these 17 patients were never seen at all.

In [ ]:
gt_masks = {}
for pid in ALL_PATIENTS:
    hits = (glob.glob(os.path.join(GT_DIR_D1,  f"{pid}*.seg.nrrd")) or
            glob.glob(os.path.join(GT_DIR_NEW, f"{pid}*.seg.nrrd")))
    if hits:
        gt_masks[pid] = hits[0]
    else:
        print(f"WARNING: No GT for {pid}")
print(f"GT masks found: {len(gt_masks)}/{len(ALL_PATIENTS)}\n")

preds_by_run = {}
for label, (pred_dir, _) in RUNS_INFO.items():
    d = patients_in_dir(pred_dir) if os.path.isdir(pred_dir) else {}
    preds_by_run[label] = d
    status = f"{len(d)}/{len(ALL_PATIENTS)}" if d else "NOT FOUND -- run inference / pull results first"
    print(f"{label:<30}: {status}")

In [ ]:
rows = []
for pid in ALL_PATIENTS:
    if pid not in gt_masks:
        continue
    gt_bin = load_nrrd_binary(gt_masks[pid])
    row = {"patient": pid, "cohort": "D1" if pid in D1_PATIENTS else "new"}
    for label, preds in preds_by_run.items():
        row[label] = (dice_score(merge_meniscus(load_vol(preds[pid])), gt_bin)
                      if pid in preds else float("nan"))
    rows.append(row)

df = pd.DataFrame(rows)
means = df[list(RUNS_INFO.keys())].mean()
print("MEAN 17pt test Dice per run (fake-only / zero-shot):")
print(means.round(4).to_string())
df

## Section 2 — Measure the Run 3 noise floor

The mixed-regime floor (0.0226) came from seed42-vs-seed7 of the *mixed* replica recipe and does
not transfer to this regime. Here it is re-measured the same way — **per-patient max |Δ| between
the two `base` seeds** — and that measured value is what every verdict below uses.

Expect it to be **larger** than 0.0226: with no real-PD anchor, training is less constrained.

In [ ]:
if df[BASE_LABEL].notna().any() and df[SEED7_LABEL].notna().any():
    pair = df[["patient", "cohort", BASE_LABEL, SEED7_LABEL]].dropna().copy()
    pair["abs_delta"] = (pair[BASE_LABEL] - pair[SEED7_LABEL]).abs()
    NOISE_FLOOR = float(pair["abs_delta"].max())
    print(f"seed42 mean = {pair[BASE_LABEL].mean():.4f}   seed7 mean = {pair[SEED7_LABEL].mean():.4f}")
    print(f"per-patient max |delta| = {NOISE_FLOOR:.4f}   <-- RUN 3 NOISE FLOOR")
    print(f"per-patient mean |delta| = {pair['abs_delta'].mean():.4f}")
    print(f"\n(mixed-regime floor was {MIXED_NOISE_FLOOR:.4f} -- shown for contrast, NOT used here)")
    display(pair.sort_values("abs_delta", ascending=False).round(4))
else:
    NOISE_FLOOR = float("nan")
    print("Both base seeds are needed to measure the Run 3 floor.")
    print("Missing:", [l for l in (BASE_LABEL, SEED7_LABEL) if df[l].isna().all()])
    print("\nNothing below can be judged until this is measured -- do NOT fall back to 0.0226.")

## Section 3 — Training-time metrics from `history.csv`

Run 3 schema. `dice_fake_va_smooth` is the selection criterion (3-epoch trailing mean), so the
"best epoch" here is the epoch `ckpt_best.pth` was actually saved at — not the raw argmax.
`dice_real_va_diag` is logged for information only and never influenced anything.

In [ ]:
def best_epoch_fakeonly(history_path):
    """Selects the row that maximises dice_fake_va_smooth -- the criterion the
    training loop itself used. Falls back to raw dice_fake_va for any run written
    with --select_smooth 1."""
    if not os.path.exists(history_path):
        return None
    rows = list(csv.DictReader(open(history_path)))
    if not rows:
        return None
    key = "dice_fake_va_smooth" if "dice_fake_va_smooth" in rows[0] else "dice_fake_va"
    best = max(rows, key=lambda r: float(r[key]))
    out = {
        "best_epoch":   int(best["epoch"]),
        "n_epochs":     len(rows),
        "dice_fake_va": float(best["dice_fake_va"]),
        "dice_fake_tr": float(best["dice_fake_tr"]),
        "smooth":       float(best[key]),
        "gap_fake":     float(best["gap_fake"]),
    }
    if "dice_real_va_diag" in best:
        out["real_va_diag"] = float(best["dice_real_va_diag"])
    for c in ("pos_wnorm", "rim_frac", "area_frac"):
        if c in best:
            out[c] = float(best[c])
    return out

val_rows = []
for label, (_, run_dir) in RUNS_INFO.items():
    info = best_epoch_fakeonly(os.path.join(run_dir, "history.csv"))
    if info is None:
        print(f"{label:<30}: history.csv not found under {run_dir}")
        continue
    val_rows.append({"run": label, **info})

val_df = pd.DataFrame(val_rows)
val_df

In [ ]:
# Did the domain gap actually close? fake-val vs the real-PD diagnostic, same checkpoint.
if not val_df.empty and "real_va_diag" in val_df.columns:
    g = val_df[["run", "dice_fake_va", "real_va_diag"]].copy()
    g["fake_minus_real"] = (g["dice_fake_va"] - g["real_va_diag"]).round(4)
    print("Domain gap per run (fake val - real val). Smaller = the idea transferred better.\n")
    display(g.round(4))
    print("Mixed-regime reference: the gap there was ~0.02 because real PD was in training.")
    print("A Run 3 gap much larger than that is expected and is the thing these ideas target.")

## Section 4 — Ranking vs the fake-only baseline, against the **measured** floor

In [ ]:
if np.isnan(NOISE_FLOOR):
    print("Run 3 noise floor not measured yet (Section 2) -- ranking withheld.")
else:
    rank_rows = []
    for label in RUNS_INFO:
        if label == SEED7_LABEL or label not in means.index or np.isnan(means[label]):
            continue
        delta = means[label] - means[BASE_LABEL]
        rank_rows.append({
            "run": label,
            "test_17pt_mean": round(means[label], 4),
            "delta_vs_base": round(delta, 4),
            "above_noise_floor": abs(delta) > NOISE_FLOOR,
            "direction": ("real gain" if delta > NOISE_FLOOR else
                          "real loss" if delta < -NOISE_FLOOR else
                          "noise / no effect"),
        })
    rank_df = pd.DataFrame(rank_rows).sort_values("test_17pt_mean", ascending=False).reset_index(drop=True)
    print(f"Baseline: {BASE_LABEL} = {means[BASE_LABEL]:.4f}   |   MEASURED floor = ±{NOISE_FLOOR:.4f}")
    display(rank_df)

    # A3 up/down read against their own control, not the plain baseline:
    ctrl = "A3-none (control)"
    if ctrl in means.index and not np.isnan(means[ctrl]):
        print(f"\nA3 secondary read -- vs its own control ({ctrl} = {means[ctrl]:.4f}):")
        for lab in ["A3-up (end slices x2)", "A3-down (end slices x0.3)"]:
            if lab in means.index and not np.isnan(means[lab]):
                d = means[lab] - means[ctrl]
                verdict = ("real gain" if d > NOISE_FLOOR else
                           "real loss" if d < -NOISE_FLOOR else "noise / no effect")
                print(f"  {lab:<30}: {means[lab]:.4f}  (delta {d:+.4f}, {verdict})")

In [ ]:
if not np.isnan(NOISE_FLOOR):
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = {"real gain": "tab:green", "real loss": "tab:red", "noise / no effect": "tab:gray"}
    ax.bar(rank_df["run"], rank_df["test_17pt_mean"], color=[colors[d] for d in rank_df["direction"]])
    b = means[BASE_LABEL]
    ax.axhline(b, linestyle="--", color="black", alpha=.6, label=f"fake-only base = {b:.4f}")
    ax.axhline(b + NOISE_FLOOR, linestyle=":", color="gray", alpha=.6)
    ax.axhline(b - NOISE_FLOOR, linestyle=":", color="gray", alpha=.6,
               label=f"±{NOISE_FLOOR:.4f} measured floor")
    ax.axhline(MIXED_REPLICA_TEST, linestyle="-.", color="tab:blue", alpha=.7,
               label=f"mixed-regime replica = {MIXED_REPLICA_TEST} (had real PD)")
    ax.set_ylabel("Mean 17pt test Dice")
    ax.set_title("Run 3 (fake-only, zero-shot) vs the measured noise floor")
    ax.set_xticks(range(len(rank_df)))
    ax.set_xticklabels(rank_df["run"], rotation=30, ha="right")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=.3)
    plt.tight_layout(); plt.show()

## Section 5 — How much of the domain gap did each idea close?

The headline framing for a zero-shot claim. `0%` = no better than fake-only training;
`100%` = matches the mixed-regime model that actually trained on 13 real patients.

In [ ]:
if not np.isnan(NOISE_FLOOR) and not np.isnan(means[BASE_LABEL]):
    span = MIXED_REPLICA_TEST - means[BASE_LABEL]
    print(f"Zero-shot baseline {means[BASE_LABEL]:.4f}  ->  mixed-regime replica {MIXED_REPLICA_TEST:.4f}")
    print(f"Domain gap to close: {span:.4f}\n")
    if span <= 0:
        print("Baseline already matches/exceeds the mixed model -- gap framing does not apply.")
    else:
        close_rows = []
        for label in RUNS_INFO:
            if label == SEED7_LABEL or label not in means.index or np.isnan(means[label]):
                continue
            d = means[label] - means[BASE_LABEL]
            close_rows.append({
                "run": label,
                "test_17pt": round(means[label], 4),
                "delta_vs_base": round(d, 4),
                "pct_of_gap_closed": round(100 * d / span, 1),
                "beats_floor": abs(d) > NOISE_FLOOR,
            })
        display(pd.DataFrame(close_rows).sort_values("pct_of_gap_closed", ascending=False).reset_index(drop=True))

## Section 6 — Cohort split (10 original D1 patients vs 7 newer)

The two cohorts were annotated at different times. A gain that appears on only one of them is
more likely annotation drift than a modelling effect — this caught three sign-flips in the
crop+sequence arm.

In [ ]:
labels = [l for l in RUNS_INFO if not df[l].isna().all()]
d1, new = df[df.cohort == "D1"], df[df.cohort == "new"]
out = []
for l in labels:
    out.append({
        "run": l,
        "D1_mean":      round(d1[l].mean(), 4),
        "new_mean":     round(new[l].mean(), 4),
        "all_17":       round(df[l].mean(), 4),
        "D1_minus_new": round(d1[l].mean() - new[l].mean(), 4),
    })
cohort_df = pd.DataFrame(out)
print(f"D1 = {len(d1)} patients, new = {len(new)} patients\n")
display(cohort_df)

if BASE_LABEL in labels and not np.isnan(NOISE_FLOOR):
    b_d1, b_new = d1[BASE_LABEL].mean(), new[BASE_LABEL].mean()
    print(f"\nDelta vs base per cohort (floor ±{NOISE_FLOOR:.4f}):")
    for _, r in cohort_df.iterrows():
        if r["run"] in (BASE_LABEL, SEED7_LABEL):
            continue
        dd1, dnew = r["D1_mean"] - b_d1, r["new_mean"] - b_new
        flip = "   <-- SIGN FLIP between cohorts" if (dd1 > 0) != (dnew > 0) else ""
        print(f"  {r['run']:<30} D1 {dd1:+.4f}   new {dnew:+.4f}{flip}")